#1-Import Libraries Make Sure Data Available

In [ ]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint


from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.svm import LinearSVC

import lightgbm as lgb


from sklearn.ensemble import ExtraTreesClassifier

from xgboost import XGBClassifier
from scipy.stats import uniform

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB



from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV


import os
for dirname, _, filenames in os.walk('/content/drive/MyDrive/GP2DarwinRF/data.csv'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

#2-Read Data Show In Table

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/GP2DarwinRF/data.csv")
df.head()

,ID,air_time1,disp_index1,gmrt_in_air1,gmrt_on_paper1,max_x_extension1,max_y_extension1,mean_acc_in_air1,mean_acc_on_paper1,mean_gmrt1,...,mean_jerk_in_air25,mean_jerk_on_paper25,mean_speed_in_air25,mean_speed_on_paper25,num_of_pendown25,paper_time25,pressure_mean25,pressure_var25,total_time25,class
0,id_1,5160,0.000013,120.804174,86.853334,957,6601,0.361800,0.217459,103.828754,...,0.141434,0.024471,5.596487,3.184589,71,40120,1749.278166,296102.7676,144605,P
1,id_2,51980,0.000016,115.318238,83.448681,1694,6998,0.272513,0.144880,99.383459,...,0.049663,0.018368,1.665973,0.950249,129,126700,1504.768272,278744.2850,298640,P
2,id_3,2600,0.000010,229.933997,172.761858,2333,5802,0.387020,0.181342,201.347928,...,0.178194,0.017174,4.000781,2.392521,74,45480,1431.443492,144411.7055,79025,P
3,id_4,2130,0.000010,369.403342,183.193104,1756,8159,0.556879,0.164502,276.298223,...,0.113905,0.019860,4.206746,1.613522,123,67945,1465.843329,230184.7154,181220,P
4,id_5,2310,0.000007,257.997131,111.275889,987,4732,0.266077,0.145104,184.636510,...,0.121782,0.020872,3.319036,1.680629,92,37285,1841.702561,158290.0255,72575,P


#3-Drop the ID column and replace class values with 1 or 0

In [ ]:
df.drop(["ID"], axis=1, inplace=True)

df['class'] = df['class'].replace({'P':1,'H':0}) #1:has probability of alzheimer's, 0:doesn't

df.head()

<ipython-input-65-f5a0b5f7c101>:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['class'] = df['class'].replace({'P':1,'H':0}) #1:has probability of alzheimer's, 0:doesn't


,air_time1,disp_index1,gmrt_in_air1,gmrt_on_paper1,max_x_extension1,max_y_extension1,mean_acc_in_air1,mean_acc_on_paper1,mean_gmrt1,mean_jerk_in_air1,...,mean_jerk_in_air25,mean_jerk_on_paper25,mean_speed_in_air25,mean_speed_on_paper25,num_of_pendown25,paper_time25,pressure_mean25,pressure_var25,total_time25,class
0,5160,0.000013,120.804174,86.853334,957,6601,0.361800,0.217459,103.828754,0.051836,...,0.141434,0.024471,5.596487,3.184589,71,40120,1749.278166,296102.7676,144605,1
1,51980,0.000016,115.318238,83.448681,1694,6998,0.272513,0.144880,99.383459,0.039827,...,0.049663,0.018368,1.665973,0.950249,129,126700,1504.768272,278744.2850,298640,1
2,2600,0.000010,229.933997,172.761858,2333,5802,0.387020,0.181342,201.347928,0.064220,...,0.178194,0.017174,4.000781,2.392521,74,45480,1431.443492,144411.7055,79025,1
3,2130,0.000010,369.403342,183.193104,1756,8159,0.556879,0.164502,276.298223,0.090408,...,0.113905,0.019860,4.206746,1.613522,123,67945,1465.843329,230184.7154,181220,1
4,2310,0.000007,257.997131,111.275889,987,4732,0.266077,0.145104,184.636510,0.037528,...,0.121782,0.020872,3.319036,1.680629,92,37285,1841.702561,158290.0255,72575,1


#4-Import Libraries, Assign X And Y, Centering X Values For PCA

In [ ]:


x = df.drop(['class'], axis=1)
y = df['class']
x_centered = x - x.mean(axis=0)

#5-Apply PCA To Take Only Most Impactful Features

In [ ]:
original_titles = x_centered.columns  # Original feature names


pca=PCA()

pca.fit(x_centered)

csum=np.cumsum(pca.explained_variance_ratio_)

d=np.argmax(csum>=0.99999)+1

d


65

#6-Split and scale Data to be in proper form to train on Models

In [ ]:
pca=PCA(n_components=d)

X2D=pca.fit_transform(x_centered)


##########

X2D_Head =  pd.DataFrame(X2D, columns=[f'PC{i+1}' for i in range(d)])

print("Dataset after PCA transformation:")
print(X2D_Head.head())

# Show original feature contributions to each principal component
loadings = pd.DataFrame(
    pca.components_.T,  # Transpose to align features with PCs
    index=original_titles,  # Original feature names
    columns=[f'PC{i+1}' for i in range(d)]  # Principal components
)

print("\nOriginal feature contributions to principal components (loadings):")
print(loadings)


# Save to an Excel file
output_file = "PCA_results.xlsx"
with pd.ExcelWriter(output_file) as writer:
    X2D_Head.to_excel(writer, sheet_name="Transformed Data", index=False)
    loadings.to_excel(writer, sheet_name="Feature Contributions")

print(f"PCA results saved to {output_file}")


##########

X_train, X_test, y_train, y_test = train_test_split(X2D, y, test_size=0.2, random_state=42)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

Dataset after PCA transformation:
             PC1            PC2           PC3            PC4            PC5  \
0 -663766.289340  -13851.602231 -37022.879317  135481.740610 -221029.569176   
1 -421640.474929  143943.491792  76977.566010  304431.745211 -346062.498913   
2 -658641.907136 -125700.920531 -37786.711548  -43958.581762   61004.327975   
3 -610505.444168   20562.058647 -27546.743363  284494.310356 -339731.388524   
4 -264898.844873 -118543.104870 -41463.351303   92972.145351 -111701.398182   

             PC6           PC7            PC8            PC9          PC10  \
0   10346.188126 -15247.088388   42417.403157   18456.089806 -19205.027889   
1  -54059.396491  21975.533596  188015.403975  -90975.481030  -5895.770594   
2  -93793.874857 -44393.160435  -13781.065935  152823.168418 -21242.178447   
3 -108457.840916 -43972.310565  -19449.959690  -44243.515454  51026.096118   
4 -113861.434283 -33322.925230  -31673.699516   92666.316239  53791.522179   

   ...          PC56  

#7-Train On RF Algorithm and evaluate model

In [ ]:

rf_clf= RandomForestClassifier(n_estimators=200,max_features='log2', min_samples_split=10, max_depth=7 , random_state=41)
linear_svc = LinearSVC(C=0.7, loss="hinge", max_iter=100000)
xgb_clf = XGBClassifier(objective ='binary:logistic', max_depth = 7,  learning_rate = 0.01,   n_estimators = 400)



svm_clf = SVC(probability=True, random_state=42)
nb_clf = GaussianNB()
log_reg = LogisticRegression(max_iter=1000, random_state=42)
# et_clf = ExtraTreesClassifier(n_estimators=100, max_depth=20, min_samples_split=2, min_samples_leaf=1, bootstrap=True , random_state=42)
# lgb_clf = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.1, random_state=42)



rf_clf.fit(X_train_scaled, y_train)
linear_svc.fit(X_train_scaled, y_train)
xgb_clf.fit(X_train_scaled, y_train)


svm_clf.fit(X_train_scaled, y_train)
log_reg.fit(X_train_scaled, y_train)
# et_clf.fit(X_train_scaled, y_train)
nb_clf.fit(X_train_scaled, y_train)

voting_clf = VotingClassifier(
    estimators=[
        ('rf', rf_clf),
        ('svm', linear_svc),
        ('xgb', xgb_clf),

        # ('log', log_reg),
        # ('et', et_clf),
    ],
    voting='hard'
)

voting_clf.fit(X_train_scaled, y_train)


# Step 1: Cross-validation
cv_scores = cross_val_score(voting_clf, X_train_scaled, y_train, cv=5, scoring='accuracy')
print("Cross-validation scores: ", cv_scores)
print("Mean cross-validation score: ", cv_scores.mean())



# Evaluate ensemble on the test set
y_pred = voting_clf.predict(X_test_scaled)

# Compute evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)  # Sensitivity
f1 = f1_score(y_test, y_pred)

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

# Specificity calculation
specificity = tn / (tn + fp)

# Print results
print("Evaluation Metrics:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall (Sensitivity): {recall:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"F1-Score: {f1:.4f}")

print("\nConfusion Matrix:")
print(cm)


# # # # Results 450 Full Feature Set
#
# features RF
# Cross-validation scores:  [0.85714286 0.92857143 0.85714286 0.78571429 0.92592593]
# Mean cross-validation score:  0.8708994708994708
# Evaluation Metrics:
# Accuracy: 0.8857
# Precision: 0.9444
# Recall (Sensitivity): 0.8500
# Specificity: 0.9333
# F1-Score: 0.8947

# Confusion Matrix:
# [[14  1]
#  [ 3 17]]
#
# Linear SVC
# Cross-validation scores:  [0.85714286 0.89285714 0.78571429 0.71428571 0.88888889]
# Mean cross-validation score:  0.8277777777777778
# Evaluation Metrics:
# Accuracy: 0.8571
# Precision: 0.8947
# Recall (Sensitivity): 0.8500
# Specificity: 0.8667
# F1-Score: 0.8718

# Confusion Matrix:
# [[13  2]
#  [ 3 17]]
#
# XGBoost
# Cross-validation scores:  [0.89285714 0.85714286 0.92857143 0.78571429 0.85185185]
# Mean cross-validation score:  0.8632275132275133
# Evaluation Metrics:
# Accuracy: 0.9143
# Precision: 0.9474
# Recall (Sensitivity): 0.9000
# Specificity: 0.9333
# F1-Score: 0.9231

# Confusion Matrix:
# [[14  1]
#  [ 2 18]]
#
# Logistic Regression
# Cross-validation scores:  [0.85714286 0.89285714 0.85714286 0.78571429 0.88888889]
# Mean cross-validation score:  0.8563492063492063
# Evaluation Metrics:
# Accuracy: 0.8571
# Precision: 0.8947
# Recall (Sensitivity): 0.8500
# Specificity: 0.8667
# F1-Score: 0.8718

# Confusion Matrix:
# [[13  2]
#  [ 3 17]]
#
# GNB
# Cross-validation scores:  [0.82142857 0.92857143 0.89285714 0.85714286 0.92592593]
# Mean cross-validation score:  0.8851851851851851
# Evaluation Metrics:
# Accuracy: 0.8286
# Precision: 0.8500
# Recall (Sensitivity): 0.8500
# Specificity: 0.8000
# F1-Score: 0.8500

# Confusion Matrix:
# [[12  3]
#  [ 3 17]]
#
# SVM
# Cross-validation scores:  [0.89285714 0.85714286 0.89285714 0.85714286 0.92592593]
# Mean cross-validation score:  0.8851851851851851
# Evaluation Metrics:
# Accuracy: 0.8286
# Precision: 0.8500
# Recall (Sensitivity): 0.8500
# Specificity: 0.8000
# F1-Score: 0.8500

# Confusion Matrix:
# [[12  3]
#  [ 3 17]]
#
# Ensemble Based On 450
# Cross-validation scores:  [0.85714286 0.92857143 0.89285714 0.82142857 0.92592593]
# Mean cross-validation score:  0.8851851851851851
# Evaluation Metrics:
# Accuracy: 0.8857
# Precision: 0.9444
# Recall (Sensitivity): 0.8500
# Specificity: 0.9333
# F1-Score: 0.8947

# Confusion Matrix:
# [[14  1]
#  [ 3 17]]
#
#
#
#
# # # # Using Only 65 Features
#
# Ensemble Results
#
# Cross-validation scores:  [0.75       0.67857143 0.85714286 0.71428571 0.77777778]
# Mean cross-validation score:  0.7555555555555555
# Evaluation Metrics:
# Accuracy: 0.9429
# Precision: 0.9500
# Recall (Sensitivity): 0.9500
# Specificity: 0.9333
# F1-Score: 0.9500

# Confusion Matrix:
# [[14  1]
#  [ 1 19]]
#
# RF Results
# Cross-validation scores:  [0.78571429 0.71428571 0.85714286 0.71428571 0.77777778]
# Mean cross-validation score:  0.7698412698412699
# Evaluation Metrics:
# Accuracy: 0.9143
# Precision: 0.9048
# Recall (Sensitivity): 0.9500
# Specificity: 0.8667
# F1-Score: 0.9268

# Confusion Matrix:
# [[13  2]
#  [ 1 19]]
#
#
# Linear SVC
# Cross-validation scores:  [0.78571429 0.67857143 0.67857143 0.64285714 0.66666667]
# Mean cross-validation score:  0.6904761904761905
# Evaluation Metrics:
# Accuracy: 0.9143
# Precision: 0.9474
# Recall (Sensitivity): 0.9000
# Specificity: 0.9333
# F1-Score: 0.9231

# Confusion Matrix:
# [[14  1]
#  [ 2 18]]
#
# XGBoost
# Cross-validation scores:  [0.71428571 0.57142857 0.75       0.67857143 0.66666667]
# Mean cross-validation score:  0.6761904761904762
# Evaluation Metrics:
# Accuracy: 0.8857
# Precision: 0.9000
# Recall (Sensitivity): 0.9000
# Specificity: 0.8667
# F1-Score: 0.9000

# Confusion Matrix:
# [[13  2]
#  [ 2 18]]
#
# Logistic Regression
#
# Cross-validation scores:  [0.82142857 0.67857143 0.75       0.67857143 0.74074074]
# Mean cross-validation score:  0.7338624338624339
# Evaluation Metrics:
# Accuracy: 0.8857
# Precision: 0.9444
# Recall (Sensitivity): 0.8500
# Specificity: 0.9333
# F1-Score: 0.8947

# Confusion Matrix:
# [[14  1]
#  [ 3 17]]
#
# GNB
# Cross-validation scores:  [0.75       0.82142857 0.78571429 0.64285714 0.77777778]
# Mean cross-validation score:  0.7555555555555555
# Evaluation Metrics:
# Accuracy: 0.8000
# Precision: 0.7600
# Recall (Sensitivity): 0.9500
# Specificity: 0.6000
# F1-Score: 0.8444

# Confusion Matrix:
# [[ 9  6]
#  [ 1 19]]
#
# SVM
# Cross-validation scores:  [0.85714286 0.75       0.85714286 0.75       0.85185185]
# Mean cross-validation score:  0.8132275132275133
# Evaluation Metrics:
# Accuracy: 0.8286
# Precision: 0.8500
# Recall (Sensitivity): 0.8500
# Specificity: 0.8000
# F1-Score: 0.8500

# Confusion Matrix:
# [[12  3]
#  [ 3 17]]

Cross-validation scores:  [0.75       0.67857143 0.85714286 0.71428571 0.77777778]
Mean cross-validation score:  0.7555555555555555
Evaluation Metrics:
Accuracy: 0.9429
Precision: 0.9500
Recall (Sensitivity): 0.9500
Specificity: 0.9333
F1-Score: 0.9500

Confusion Matrix:
[[14  1]
 [ 1 19]]


#8-final evaluation of model

In [ ]:
# Evaluate the final model on the test set
test_accuracy = best_rf.score(X_test_scaled, y_test)
print(f"Test set accuracy: {test_accuracy:.4f}")

NameError: name 'best_rf' is not defined